<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Lab6_Text_Representation_and_Retrieval_Foundations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6: Text Representation and Retrieval Foundations
## From cleaned text to searchable knowledge

In the previous lab, we learned how to clean and normalize text.

In this lab, we answer the next important question:

> After text is cleaned, how can a computer compare a user query with many documents and retrieve the most relevant ones?


## The big picture

A retrieval system usually follows this pipeline:

```text
Raw documents
→ optional preprocessing
→ tokenization
→ vocabulary construction
→ document vectors
→ query vector
→ similarity or ranking score
→ top-k retrieved documents
→ evaluation
```

This lab focuses on **lexical retrieval**.

Lexical retrieval means the system mainly depends on exact word overlap between the query and the documents.

Examples of lexical methods:

- Bag-of-Words
- TF-IDF
- BM25

These methods do **not** truly understand meaning. If the query says "money back" and the document says "refund", a lexical method may fail because the words are different.

That weakness is exactly why later labs should introduce embeddings and semantic retrieval.


## Learning outcomes

After completing this lab, you should be able to explain and implement:

| Topic | What you must understand |
|---|---|
| Corpus | A collection of documents |
| Query | The user's search question |
| Token | A basic unit of text, usually a word |
| Vocabulary | The set of unique terms learned from the corpus |
| Document-term matrix | A matrix where rows are documents and columns are words |
| Sparse matrix | A matrix with mostly zero values |
| Bag-of-Words | Text representation using word counts |
| N-grams | Consecutive word sequences such as one-word or two-word features |
| TF | Term frequency |
| DF | Document frequency |
| IDF | Inverse document frequency |
| TF-IDF | A weighted representation of words based on frequency and rarity |
| Cosine similarity | A way to compare vectors by direction |
| Top-k retrieval | Returning the highest-scoring k documents |
| Ground truth | Human-defined relevant documents for each query |
| Precision@K | How many returned documents are relevant |
| Recall@K | How many relevant documents were found |
| Hit Rate@K | Whether at least one relevant document was found |
| Reciprocal Rank and MRR | How early the first relevant document appears |
| BM25 | A stronger keyword ranking method with length normalization |
| Chunking | Splitting long documents into smaller retrievable pieces |


## Section 1 — Import libraries

We will use a small set of libraries.

### `re`
Python's regular expression module. We use it for simple tokenization and cleaning.

### `CountVectorizer`
Converts documents into word-count vectors.

### `TfidfVectorizer`
Converts documents into TF-IDF vectors.

### `cosine_similarity`
Compares vectors using cosine similarity.



In [ ]:
import re
import math
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 120)


## Section 2 — Create a neutral knowledge base

We need a small corpus.

A **corpus** is a collection of documents.

In real systems, a corpus may contain:

- help center articles,
- policy documents,
- product manuals,
- lecture notes,
- research abstracts,
- legal documents,
- medical guidelines,
- internal company documentation.

For this lab, we will use a neutral student-services knowledge base.

Each sentence below is treated as one document.


In [ ]:
documents = [
    "The library account allows students to borrow books, renew loans, and reserve study rooms.",
    "Password reset requires entering your university email and verifying a code sent to your phone.",
    "The writing center helps students improve essays, citations, and research proposals.",
    "The campus health clinic provides vaccinations, first aid, and appointment booking.",
    "Students can request a tuition refund within 14 days of course withdrawal.",
    "The data science workshop introduces Python, pandas, machine learning, and model evaluation.",
    "The robotics workshop covers sensors, motors, control systems, and autonomous navigation.",
    "The research database provides access to journal articles, conference papers, and citation tools.",
    "Two-factor authentication protects accounts by requiring a password and a temporary security code.",
    "The printing service supports color printing, black-and-white printing, scanning, and payment by student card.",
    "The academic calendar lists registration dates, exam weeks, holidays, and withdrawal deadlines.",
    "The internship office helps students prepare resumes, apply for internships, and schedule career advising."
]

corpus_df = pd.DataFrame({
    "document_id": range(len(documents)),
    "document": documents
})

corpus_df


,document_id,document
0,0,"The library account allows students to borrow books, renew loans, and reserve study rooms."
1,1,Password reset requires entering your university email and verifying a code sent to your phone.
2,2,"The writing center helps students improve essays, citations, and research proposals."
3,3,"The campus health clinic provides vaccinations, first aid, and appointment booking."
4,4,Students can request a tuition refund within 14 days of course withdrawal.
5,5,"The data science workshop introduces Python, pandas, machine learning, and model evaluation."
6,6,"The robotics workshop covers sensors, motors, control systems, and autonomous navigation."
7,7,"The research database provides access to journal articles, conference papers, and citation tools."
8,8,Two-factor authentication protects accounts by requiring a password and a temporary security code.
9,9,"The printing service supports color printing, black-and-white printing, scanning, and payment by student card."


## How to read the corpus table

- `document_id` is the unique ID of the document.
- `document` is the original text.

Retrieval systems usually work with IDs internally.

Example:

If the system returns `document_id = 4`, we can look up the original text:

```text
Students can request a tuition refund within 14 days of course withdrawal.
```


## Section 3 — Create user queries

A **query** is the user's input to the retrieval system.

In a search engine, the query may be short:

```text
password reset
```

In a question-answering system, the query may be a full question:

```text
How do I reset my password?
```

The retrieval system should compare the query with all documents and return the best matches.


In [ ]:
queries = [
    "How do I reset my password?",
    "Where can I borrow books?",
    "How can I get a refund?",
    "What workshop teaches machine learning?",
    "Where can I find journal articles?",
    "How do I print in color?",
    "What office helps with internships?",
    "What protects my account with a security code?",
    "Where can I check exam weeks and holidays?",
    "What workshop covers sensors and motors?"
]

queries_df = pd.DataFrame({
    "query_id": range(len(queries)),
    "query": queries
})

queries_df


,query_id,query
0,0,How do I reset my password?
1,1,Where can I borrow books?
2,2,How can I get a refund?
3,3,What workshop teaches machine learning?
4,4,Where can I find journal articles?
5,5,How do I print in color?
6,6,What office helps with internships?
7,7,What protects my account with a security code?
8,8,Where can I check exam weeks and holidays?
9,9,What workshop covers sensors and motors?


## Section 4 — Bag-of-Words concept

Bag-of-Words is one of the simplest text representations.

It keeps:

- which words appear,
- how many times each word appears.

It ignores:

- grammar,
- word order,
- sentence structure,
- meaning beyond exact words.

### Mathematical definition

Let:

- $D$ = number of documents
- $V$ = number of unique vocabulary terms
- $X$ = document-term matrix

Then:

$$
X_{i,j} = \text{count of term } j \text{ in document } i
$$

So the matrix shape is:

$$
D \times V
$$

Rows are documents. Columns are terms. Values are counts.


## When to use Bag-of-Words

Use Bag-of-Words when:

- you need a simple baseline,
- you want interpretable word-count features,
- the task depends heavily on exact keyword occurrence,
- you are teaching or debugging text representation.

Do not expect Bag-of-Words to understand synonyms.

Example failure:

```text
Query: money back
Document: tuition refund
```

A Bag-of-Words model may not connect these two phrases because they use different words.


## Section 5 — A tiny manual example before using libraries

Before using `CountVectorizer`, we need to understand the idea manually.

Suppose we have two documents:

```text
Document 0: password reset code
Document 1: password security code
```

The vocabulary is:

```text
["code", "password", "reset", "security"]
```

Now convert each document into counts:

| document | code | password | reset | security |
|---|---:|---:|---:|---:|
| Document 0 | 1 | 1 | 1 | 0 |
| Document 1 | 1 | 1 | 0 | 1 |

This is called a **document-term matrix**.


## Section 6 — `CountVectorizer`

`CountVectorizer` is a scikit-learn class that converts raw text into a document-term matrix.

It does four main things:

1. Tokenizes the text.
2. Builds a vocabulary.
3. Counts term occurrences.
4. Returns a sparse document-term matrix.

Important methods:

| Method | Meaning |
|---|---|
| `fit()` | Learn the vocabulary from training documents |
| `transform()` | Convert text into vectors using an already learned vocabulary |
| `fit_transform()` | Learn the vocabulary and convert the same documents at once |
| `get_feature_names_out()` | Return vocabulary terms as column names |

For the corpus, we use `fit_transform()` because the vectorizer has not learned vocabulary yet.


In [ ]:
count_vectorizer = CountVectorizer()

bow_matrix = count_vectorizer.fit_transform(documents)

vocabulary = count_vectorizer.get_feature_names_out()

bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vocabulary
)

bow_df.insert(0, "document_id", range(len(documents)))

bow_df


,document_id,14,academic,access,account,accounts,advising,aid,allows,and,apply,appointment,articles,authentication,autonomous,black,booking,books,borrow,by,calendar,campus,can,card,career,center,citation,citations,clinic,code,color,conference,control,course,covers,data,database,dates,days,deadlines,email,entering,essays,evaluation,exam,factor,first,for,health,helps,holidays,improve,internship,internships,introduces,journal,learning,library,lists,loans,machine,model,motors,navigation,of,office,pandas,papers,password,payment,phone,prepare,printing,proposals,protects,provides,python,refund,registration,renew,request,requires,requiring,research,reserve,reset,resumes,robotics,rooms,scanning,schedule,science,security,sensors,sent,service,student,students,study,supports,systems,temporary,the,to,tools,tuition,two,university,vaccinations,verifying,weeks,white,withdrawal,within,workshop,writing,your
0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,2
2,2,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0
3,3,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0
4,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0
5,5,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
6,6,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0
7,7,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
8,8,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
9,9,0,0,0,0,0,0,0,0,2,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0


## Explanation of the previous code

### `CountVectorizer()`
Creates a vectorizer object.

At this moment, it has not learned any vocabulary yet.

### `fit_transform(documents)`
This does two operations:

1. **Fit**: learn all vocabulary terms from `documents`.
2. **Transform**: convert each document into a word-count vector.

### `get_feature_names_out()`
Returns the vocabulary learned by the vectorizer.

### `toarray()`
The original output is a sparse matrix. We convert it to a normal dense array only for display.

In real large systems, converting sparse matrices to dense arrays can waste memory.

### `pd.DataFrame(...)`
Turns the matrix into a readable table.


## Section 7 — Vocabulary

The **vocabulary** is the set of unique tokens learned from the corpus.

Each vocabulary term becomes one column in the document-term matrix.

If the vocabulary has 10,000 terms, each document vector has 10,000 dimensions.

This is why classical text features can become very high-dimensional.


In [ ]:
print("Vocabulary size:", len(vocabulary))
print("Matrix shape:", bow_matrix.shape)
print("First 40 vocabulary terms:")
print(vocabulary[:40])


Vocabulary size: 116
Matrix shape: (12, 116)
First 40 vocabulary terms:
['14' 'academic' 'access' 'account' 'accounts' 'advising' 'aid' 'allows'
 'and' 'apply' 'appointment' 'articles' 'authentication' 'autonomous'
 'black' 'booking' 'books' 'borrow' 'by' 'calendar' 'campus' 'can' 'card'
 'career' 'center' 'citation' 'citations' 'clinic' 'code' 'color'
 'conference' 'control' 'course' 'covers' 'data' 'database' 'dates' 'days'
 'deadlines' 'email']


## Section 8 — Sparse matrices

A sparse matrix is a matrix where most values are zero.

Text matrices are usually sparse because each document contains only a small subset of the full vocabulary.

Example:

If the vocabulary contains 20,000 words and one document contains 80 unique words, then most of the 20,000 vector positions are zero.

Sparse storage saves memory by storing only non-zero values.


In [ ]:
total_values = bow_matrix.shape[0] * bow_matrix.shape[1]
non_zero_values = bow_matrix.count_nonzero()
zero_values = total_values - non_zero_values
sparsity_percentage = (zero_values / total_values) * 100

print("Total matrix values:", total_values)
print("Non-zero values:", non_zero_values)
print("Zero values:", zero_values)
print("Sparsity percentage:", round(sparsity_percentage, 2), "%")


Total matrix values: 1392
Non-zero values: 147
Zero values: 1245
Sparsity percentage: 89.44 %


## Why sparsity matters

Sparse matrices matter because real retrieval systems may contain:

- millions of documents,
- hundreds of thousands of terms,
- billions of possible matrix cells.

Most of those cells are zero.


## Section 9 — Inspect one document vector

To understand Bag-of-Words, inspect one document and its non-zero features.


In [ ]:
doc_id = 1

print("Original document:")
print(documents[doc_id])

row = bow_matrix[doc_id].toarray().flatten()

non_zero_terms = pd.DataFrame({
    "term": vocabulary,
    "count": row
})

non_zero_terms = non_zero_terms[non_zero_terms["count"] > 0]
non_zero_terms.sort_values(by="count", ascending=False)


Original document:
Password reset requires entering your university email and verifying a code sent to your phone.


,term,count
115,your,2
8,and,1
28,code,1
39,email,1
40,entering,1
67,password,1
69,phone,1
80,requires,1
84,reset,1
93,sent,1


## Interpretation

The table shows only words that appear in the selected document.

If a word has count `1`, it appears once.

If a word has count `2`, it appears twice.

This is simple and interpretable, but it is not enough for strong retrieval.


## Section 10 — Important `CountVectorizer` parameters

`CountVectorizer` has many parameters. We will focus on the ones students are most likely to use.

| Parameter | Meaning | When to use it |
|---|---|---|
| `lowercase=True` | Convert text to lowercase before tokenizing | Usually useful for normalizing text |
| `stop_words='english'` | Remove common English stopwords | Useful for keyword search, dangerous if stopwords carry meaning |
| `ngram_range=(1, 2)` | Use unigrams and bigrams | Useful when phrases matter |


We will demonstrate the effect of some of these parameters.


## Section 11 — Stopword removal inside vectorizers

Stopwords are common words such as:

```text
the, is, and, in, of, to
```

They may add noise in some retrieval tasks.

However, removing stopwords can be dangerous.

Example:

```text
"not allowed"
```

If `not` is removed, the meaning changes.

So stopword removal is not automatically good. It must be tested.


In [ ]:
vectorizer_no_stopwords = CountVectorizer(stop_words="english")
stopword_matrix = vectorizer_no_stopwords.fit_transform(documents)
stopword_vocab = vectorizer_no_stopwords.get_feature_names_out()

print("Vocabulary size without stopword removal:", len(vocabulary))
print("Vocabulary size with English stopword removal:", len(stopword_vocab))
print("First 40 terms after stopword removal:")
print(stopword_vocab[:40])


Vocabulary size without stopword removal: 116
Vocabulary size with English stopword removal: 105
First 40 terms after stopword removal:
['14' 'academic' 'access' 'account' 'accounts' 'advising' 'aid' 'allows'
 'apply' 'appointment' 'articles' 'authentication' 'autonomous' 'black'
 'booking' 'books' 'borrow' 'calendar' 'campus' 'card' 'career' 'center'
 'citation' 'citations' 'clinic' 'code' 'color' 'conference' 'control'
 'course' 'covers' 'data' 'database' 'dates' 'days' 'deadlines' 'email'
 'entering' 'essays' 'evaluation']


## Section 12 — N-grams

A unigram is a single word.

Example:

```text
password
```

A bigram is a sequence of two consecutive words.

Example:

```text
password reset
```

A trigram is a sequence of three consecutive words.

Example:

```text
reset your password
```

### Why n-grams matter

Sometimes a phrase means more than individual words.

Example:

```text
security code
```

The word `security` alone is broad.
The word `code` alone is broad.
The phrase `security code` is more specific.


In [ ]:
ngram_vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words="english")
ngram_matrix = ngram_vectorizer.fit_transform(documents)
ngram_vocab = ngram_vectorizer.get_feature_names_out()

print("Vocabulary size with unigrams only:", len(stopword_vocab))
print("Vocabulary size with unigrams + bigrams:", len(ngram_vocab))
print("Sample n-gram features:")
print(ngram_vocab[:80])


Vocabulary size with unigrams only: 105
Vocabulary size with unigrams + bigrams: 209
Sample n-gram features:
['14' '14 days' 'academic' 'academic calendar' 'access' 'access journal'
 'account' 'account allows' 'accounts' 'accounts requiring' 'advising'
 'aid' 'aid appointment' 'allows' 'allows students' 'apply'
 'apply internships' 'appointment' 'appointment booking' 'articles'
 'articles conference' 'authentication' 'authentication protects'
 'autonomous' 'autonomous navigation' 'black' 'black white' 'booking'
 'books' 'books renew' 'borrow' 'borrow books' 'calendar' 'calendar lists'
 'campus' 'campus health' 'card' 'career' 'career advising' 'center'
 'center helps' 'citation' 'citation tools' 'citations'
 'citations research' 'clinic' 'clinic provides' 'code' 'code sent'
 'color' 'color printing' 'conference' 'conference papers' 'control'
 'control systems' 'course' 'course withdrawal' 'covers' 'covers sensors'
 'data' 'data science' 'database' 'database provides' 'dates' 'dates exa

## N-gram warning

N-grams increase feature size.

If you use:

```python
ngram_range=(1, 3)
```

you include:

- unigrams,
- bigrams,
- trigrams.

This can improve phrase matching, but it can also create a much larger vocabulary.

Professional rule:

> Use n-grams when phrases matter, but monitor vocabulary size and retrieval performance.


## Section 13 — Why Bag-of-Words is not enough

Raw word counts have several weaknesses:

1. Common words can dominate.
2. Repeated words may get too much influence.
3. Rare important words are not emphasized enough.
4. Long documents may naturally have larger counts.
5. Synonyms are not understood.

Example:

```text
Document A: password password password password password
Document B: password reset security code
```

Bag-of-Words gives Document A a high count for `password`, but Document B may be more useful for a query about password reset.

This motivates TF-IDF.


In [ ]:
toy_docs = [
    "password password password password password",
    "password reset security code"
]

toy_count_vectorizer = CountVectorizer()
toy_counts = toy_count_vectorizer.fit_transform(toy_docs)

toy_count_df = pd.DataFrame(
    toy_counts.toarray(),
    columns=toy_count_vectorizer.get_feature_names_out()
)

toy_count_df.insert(0, "document", ["A", "B"])
toy_count_df


,document,code,password,reset,security
0,A,0,5,0,0
1,B,1,1,1,1


## Section 14 — TF-IDF concept

TF-IDF means:

```text
Term Frequency - Inverse Document Frequency
```

It combines two ideas:

### 1. Term Frequency

How often does a term appear in a document?

### 2. Inverse Document Frequency

How rare is the term across the full corpus?

A word receives a high TF-IDF score when:

- it appears in the current document,
- and it does not appear in many other documents.

A word receives a low TF-IDF score when:

- it appears in almost every document,
- or it does not appear in the current document.


## Section 15 — TF-IDF math

The core idea is:

$$
\text{TF-IDF}(t,d) = \text{TF}(t,d) \times \text{IDF}(t)
$$

Where:

- $t$ = term
- $d$ = document
- $\text{TF}(t,d)$ = frequency of term $t$ in document $d$
- $\text{IDF}(t)$ = inverse document frequency of term $t$

A common smoothed IDF formula is:

$$
\text{IDF}(t) = \log\left(\frac{N}{\text{DF}(t)}\right)
$$

Where:

- $N$ = total number of documents
- $\text{DF}(t)$ = number of documents containing term $t$




## Section 16 — Manual TF-IDF intuition

Suppose there are 12 documents.

The term `students` appears in 4 documents.

The term `vaccinations` appears in 1 document.

Then `vaccinations` should receive a higher IDF because it is more specific.

Rare terms are not always important, but they are often more discriminative for retrieval.


In [ ]:
# Compare IDF values using a vectorizer without normalization to make inspection easier.
inspection_tfidf = TfidfVectorizer(norm=None)
inspection_tfidf_matrix = inspection_tfidf.fit_transform(documents)
inspection_vocab = inspection_tfidf.get_feature_names_out()

idf_df = pd.DataFrame({
    "term": inspection_vocab,
    "idf": inspection_tfidf.idf_
}).sort_values(by="idf", ascending=False)

idf_df.head(20)


,term,idf
0,14,2.871802
69,phone,2.871802
81,requiring,2.871802
80,requires,2.871802
79,request,2.871802
78,renew,2.871802
77,registration,2.871802
76,refund,2.871802
75,python,2.871802
73,protects,2.871802


## How to interpret IDF values

Higher IDF means the term appears in fewer documents.

Lower IDF means the term appears in more documents.

In retrieval, high-IDF terms can help distinguish one document from another.

But be careful:

- Rare typos can also receive high IDF.
- Rare irrelevant tokens can also receive high IDF.
- IDF does not understand meaning; it only measures corpus-level rarity.


## Section 17 — `TfidfVectorizer`

`TfidfVectorizer` converts raw text into TF-IDF vectors.

It is approximately equivalent to:

```text
CountVectorizer + TfidfTransformer
```

It performs:

1. Tokenization.
2. Vocabulary construction.
3. Count calculation.
4. TF-IDF weighting.
5. Optional normalization.

We will now create TF-IDF vectors for the corpus.


In [ ]:
tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

tfidf_vocabulary = tfidf_vectorizer.get_feature_names_out()

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf_vocabulary
)

tfidf_df.insert(0, "document_id", range(len(documents)))

tfidf_df


,document_id,14,academic,access,account,accounts,advising,aid,allows,and,apply,appointment,articles,authentication,autonomous,black,booking,books,borrow,by,calendar,campus,can,card,career,center,citation,citations,clinic,code,color,conference,control,course,covers,data,database,dates,days,deadlines,email,entering,essays,evaluation,exam,factor,first,for,health,helps,holidays,improve,internship,internships,introduces,journal,learning,library,lists,loans,machine,model,motors,navigation,of,office,pandas,papers,password,payment,phone,prepare,printing,proposals,protects,provides,python,refund,registration,renew,request,requires,requiring,research,reserve,reset,resumes,robotics,rooms,scanning,schedule,science,security,sensors,sent,service,student,students,study,supports,systems,temporary,the,to,tools,tuition,two,university,vaccinations,verifying,weeks,white,withdrawal,within,workshop,writing,your
0,0,0.000000,0.000000,0.000000,0.296514,0.000000,0.000000,0.000000,0.296514,0.111515,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.296514,0.296514,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.296514,0.000000,0.296514,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.296514,0.000000,0.000000,0.000000,0.000000,0.296514,0.000000,0.000000,0.000000,0.296514,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.201907,0.296514,0.000000,0.000000,0.000000,0.130340,0.224947,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.099831,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.227968,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.265446,0.265446,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.227968,0.000000,0.265446,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.265446,0.000000,0.000000,0.000000,0.265446,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.265446,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.201377,0.000000,0.000000,0.000000,0.265446,0.000000,0.265446,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.530893
2,2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.130750,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.347662,0.000000,0.347662,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.347662,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.298576,0.000000,0.347662,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.347662,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.298576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.236735,0.000000,0.000000,0.000000,0.000000,0.152822,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000

## Explanation of the previous code

### `TfidfVectorizer()`
Creates a vectorizer that can convert text into TF-IDF features.

### `fit_transform(documents)`
Learns:

- vocabulary,
- document frequencies,
- IDF values.

Then it transforms each document into a TF-IDF vector.

### `tfidf_matrix`
A sparse matrix where:

- rows are documents,
- columns are vocabulary terms,
- values are TF-IDF weights.


## Section 18 — Inspect important terms in one document


We will check the highest TF-IDF terms for one document.


In [ ]:
def show_top_tfidf_terms(document_id, top_n=10):
    row = tfidf_matrix[document_id].toarray().flatten()

    scores_df = pd.DataFrame({
        "term": tfidf_vocabulary,
        "tfidf_score": row
    })

    scores_df = scores_df[scores_df["tfidf_score"] > 0]
    scores_df = scores_df.sort_values(by="tfidf_score", ascending=False)

    print("Document ID:", document_id)
    print("Document:", documents[document_id])
    return scores_df.head(top_n)

show_top_tfidf_terms(document_id=1, top_n=10)


Document ID: 1
Document: Password reset requires entering your university email and verifying a code sent to your phone.


,term,tfidf_score
115,your,0.530893
39,email,0.265446
40,entering,0.265446
69,phone,0.265446
80,requires,0.265446
84,reset,0.265446
93,sent,0.265446
106,university,0.265446
108,verifying,0.265446
28,code,0.227968


## Interpretation

The top terms should match the specific topic of the document.

For the password reset document, terms such as:

- password,
- reset,
- email,
- phone,
- verifying,
- code

should receive meaningful weights.

If the top terms look irrelevant, your preprocessing or vectorization choices may be wrong.


## Section 19 — Query vectorization

A query must be converted into the same vector space as the documents.

This is a critical rule:

```text
Documents: fit_transform()
Query: transform()
```

Why?

Because the vectorizer already learned the vocabulary from the documents. The query must use that same vocabulary.

If you call `fit_transform()` on the query, it learns a new vocabulary from the query only. Then the query vector and document vectors are not comparable.


In [ ]:
query = "How do I reset my password?"

query_vector = tfidf_vectorizer.transform([query])

print("Query vector shape:", query_vector.shape)
print("Document matrix shape:", tfidf_matrix.shape)


Query vector shape: (1, 116)
Document matrix shape: (12, 116)


## Section 20 — Out-of-vocabulary terms

An out-of-vocabulary term is a word that appears in the query but was not learned from the document corpus.

Example:

If the query contains:

```text
money back
```

but the corpus never contains `money` or `back`, those words cannot contribute to TF-IDF similarity.

Lexical retrieval depends heavily on vocabulary overlap.


In [ ]:
def inspect_query_terms(query, vectorizer):
    analyzer = vectorizer.build_analyzer()
    query_terms = analyzer(query)
    vocab_set = set(vectorizer.get_feature_names_out())

    rows = []
    for term in query_terms:
        rows.append({
            "term": term,
            "in_vocabulary": term in vocab_set
        })

    return pd.DataFrame(rows)

inspect_query_terms("How can I get my money back?", tfidf_vectorizer)


,term,in_vocabulary
0,how,False
1,can,True
2,get,False
3,my,False
4,money,False
5,back,False


## Interpretation

If important query terms are not in the vocabulary, the retriever may fail.

This is a lexical retrieval limitation, not a coding bug.

Later, semantic embeddings can reduce this problem because they can represent related meanings even when words are different.


## Section 21 — Cosine similarity concept

After representing documents and queries as vectors, we need a way to compare them.

Cosine similarity compares the **direction** of two vectors.

It is widely used in text retrieval because we often care more about term pattern than raw vector length.

### Formula

$$
\cos(\theta) = \frac{A \cdot B}{\lVert A \rVert \lVert B \rVert}
$$

Where:

- $A \cdot B$ = dot product of vectors $A$ and $B$
- $\lVert A \rVert$ = length of vector $A$
- $\lVert B \rVert$ = length of vector $B$

### Score interpretation

| Score | Meaning |
|---:|---|
| 1.0 | Very similar direction |
| 0.0 | No similarity in this vector space |
| -1.0 | Opposite direction, uncommon for TF-IDF because values are non-negative |


## Section 22 — Manual cosine similarity example

Let:

$$
A = [1, 1, 0]
$$

$$
B = [1, 0, 1]
$$

Dot product:

$$
A \cdot B = (1)(1) + (1)(0) + (0)(1) = 1
$$

Vector lengths:

$$
\lVert A \rVert = \sqrt{1^2 + 1^2 + 0^2} = \sqrt{2}
$$

$$
\lVert B \rVert = \sqrt{1^2 + 0^2 + 1^2} = \sqrt{2}
$$

Cosine similarity:

$$
\frac{1}{\sqrt{2}\sqrt{2}} = \frac{1}{2} = 0.5
$$


## Section 23 — Calculate query-document similarity

Now we compare one query vector against all document vectors.

The output will be one similarity score per document.


In [ ]:
query = "How do I reset my password?"
query_vector = tfidf_vectorizer.transform([query])

similarity_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

similarity_scores


array([0.        , 0.34990204, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.17388808, 0.        ,
       0.        , 0.        ])

## Explanation of the previous code

### `cosine_similarity(query_vector, tfidf_matrix)`
Compares the query vector with every document vector.

The result shape before flattening is:

```text
(1, number_of_documents)
```

### `.flatten()`
Converts the result into a one-dimensional array.

That makes it easier to put the scores into a table.


In [ ]:
results_df = pd.DataFrame({
    "document_id": range(len(documents)),
    "document": documents,
    "similarity_score": similarity_scores
})

results_df = results_df.sort_values(by="similarity_score", ascending=False)

results_df


,document_id,document,similarity_score
1,1,Password reset requires entering your university email and verifying a code sent to your phone.,0.349902
8,8,Two-factor authentication protects accounts by requiring a password and a temporary security code.,0.173888
0,0,"The library account allows students to borrow books, renew loans, and reserve study rooms.",0.000000
2,2,"The writing center helps students improve essays, citations, and research proposals.",0.000000
3,3,"The campus health clinic provides vaccinations, first aid, and appointment booking.",0.000000
4,4,Students can request a tuition refund within 14 days of course withdrawal.,0.000000
5,5,"The data science workshop introduces Python, pandas, machine learning, and model evaluation.",0.000000
6,6,"The robotics workshop covers sensors, motors, control systems, and autonomous navigation.",0.000000
7,7,"The research database provides access to journal articles, conference papers, and citation tools.",0.000000
9,9,"The printing service supports color printing, black-and-white printing, scanning, and payment by student card.",0.000000


## Interpretation

The best result should be the document about password reset.

This means our system has performed a basic retrieval operation:

```text
query → vector → compare with documents → sort by similarity
```

This is the core of many retrieval systems.


## Section 24 — Top-K retrieval

Top-K retrieval means returning only the best `K` documents.

If:

```text
K = 3
```

then we return the top 3 documents.

### Why K matters

Small K:

- less noise,
- but may miss useful information.

Large K:

- more chance to include relevant documents,
- but may include irrelevant documents.

Professional rule:

> K is not random. It must be tested and evaluated.


In [ ]:
top_k = 3
results_df.head(top_k)


,document_id,document,similarity_score
1,1,Password reset requires entering your university email and verifying a code sent to your phone.,0.349902
8,8,Two-factor authentication protects accounts by requiring a password and a temporary security code.,0.173888
0,0,"The library account allows students to borrow books, renew loans, and reserve study rooms.",0.000000


## Section 25 — Build a reusable TF-IDF retriever

Now we build a function so we can reuse the retrieval logic.



In [ ]:
def retrieve_top_k_tfidf(query, vectorizer, document_matrix, original_documents, k=3):
    # Retrieve the top-k most similar documents using TF-IDF and cosine similarity.
    # Parameters:
    # - query: user search query as a string.
    # - vectorizer: fitted TfidfVectorizer.
    # - document_matrix: TF-IDF matrix of all documents.
    # - original_documents: original document texts.
    # - k: number of top results to return.

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(query_vector, document_matrix).flatten()

    results = pd.DataFrame({
        "document_id": range(len(original_documents)),
        "document": original_documents,
        "score": scores
    })

    results = results.sort_values(by="score", ascending=False)

    return results.head(k).reset_index(drop=True)


In [ ]:
retrieve_top_k_tfidf(
    query="What workshop teaches machine learning?",
    vectorizer=tfidf_vectorizer,
    document_matrix=tfidf_matrix,
    original_documents=documents,
    k=3
)


,document_id,document,score
0,5,"The data science workshop introduces Python, pandas, machine learning, and model evaluation.",0.521338
1,6,"The robotics workshop covers sensors, motors, control systems, and autonomous navigation.",0.147999
2,0,"The library account allows students to borrow books, renew loans, and reserve study rooms.",0.000000


## Function explanation

### `query_vector = vectorizer.transform([query])`
Converts the query into the same TF-IDF space as the documents.

### `cosine_similarity(...)`
Calculates similarity between the query and every document.



## Section 26 — Ground truth

To evaluate retrieval, we need ground truth.

**Ground truth** means the correct relevant documents according to human judgment.

For each query, we manually define which document IDs should count as relevant.

Without ground truth, we can only guess whether the retriever is good.


In [ ]:
ground_truth = {
    "How do I reset my password?": [1, 8],
    "Where can I borrow books?": [0],
    "How can I get a refund?": [4],
    "What workshop teaches machine learning?": [5],
    "Where can I find journal articles?": [7],
    "How do I print in color?": [9],
    "What office helps with internships?": [11],
    "What protects my account with a security code?": [8, 1],
    "Where can I check exam weeks and holidays?": [10],
    "What workshop covers sensors and motors?": [6]
}

ground_truth_df = pd.DataFrame([
    {"query": query, "relevant_document_ids": relevant_ids}
    for query, relevant_ids in ground_truth.items()
])

ground_truth_df


,query,relevant_document_ids
0,How do I reset my password?,"[1, 8]"
1,Where can I borrow books?,[0]
2,How can I get a refund?,[4]
3,What workshop teaches machine learning?,[5]
4,Where can I find journal articles?,[7]
5,How do I print in color?,[9]
6,What office helps with internships?,[11]
7,What protects my account with a security code?,"[8, 1]"
8,Where can I check exam weeks and holidays?,[10]
9,What workshop covers sensors and motors?,[6]


## How to create ground truth properly

Ground truth should be created by reading the documents and deciding relevance.

Bad ground truth leads to bad evaluation.

Rules:

1. Do not mark a document relevant just because it shares one word.
2. Mark a document relevant if it helps answer the query.
3. Allow more than one relevant document if multiple documents are useful.
4. Keep document IDs stable.
5. Review ground truth if documents change.


## Section 27 — Retrieval metrics overview

We will evaluate retrieval using ranking metrics.

The most important metrics in this lab are:

| Metric | Main question |
|---|---|
| Precision@K | Of the top K retrieved documents, how many are relevant? |
| Recall@K | Of all relevant documents, how many did we retrieve in top K? |
| Hit Rate@K | Did we retrieve at least one relevant document in top K? |
| Reciprocal Rank | How early did the first relevant result appear? |
| MRR | Average reciprocal rank across all queries |

These are not classification metrics. They are retrieval/ranking metrics.


## Section 28 — Precision@K

Precision@K asks:

> Out of the top K retrieved documents, how many are relevant?

Formula:

$$
\text{Precision@K} = \frac{|\text{Retrieved@K} \cap \text{Relevant}|}{K}
$$

Example:

```text
Retrieved top 3: [1, 8, 4]
Relevant:        [1, 8]
Correct hits:    [1, 8]
```

$$
\text{Precision@3} = \frac{2}{3} = 0.67
$$

Use Precision@K when you care about avoiding irrelevant results.


## Section 29 — Recall@K

Recall@K asks:

> Out of all relevant documents, how many did the system retrieve in the top K?

Formula:

$$
\text{Recall@K} = \frac{|\text{Retrieved@K} \cap \text{Relevant}|}{|\text{Relevant}|}
$$

Example:

```text
Retrieved top 3: [1, 4, 5]
Relevant:        [1, 8]
Correct hits:    [1]
```

$$
\text{Recall@3} = \frac{1}{2} = 0.5
$$

Use Recall@K when missing relevant documents is costly.


## Section 30 — Hit Rate@K

Hit Rate@K asks:

> Did the system retrieve at least one relevant document in the top K?

Formula:

$$
\text{Hit Rate@K} =
\begin{cases}
1, & \text{if } |\text{Retrieved@K} \cap \text{Relevant}| > 0 \\
0, & \text{otherwise}
\end{cases}
$$

Example:

```text
Retrieved top 3: [4, 7, 1]
Relevant:        [1, 8]
```

There is at least one hit: document 1.

So:

$$
\text{Hit Rate@3} = 1
$$

Use Hit Rate@K when one good result is enough.


## Section 31 — Reciprocal Rank and MRR

Reciprocal Rank focuses on the position of the first relevant result.

Formula for one query:

$$
\text{Reciprocal Rank} = \frac{1}{\text{rank of first relevant result}}
$$

Example:

```text
Retrieved results: [7, 1, 8]
Relevant:          [1, 8]
```

The first relevant result is document 1 at rank 2.

$$
\text{RR} = \frac{1}{2} = 0.5
$$

MRR means Mean Reciprocal Rank:

$$
\text{MRR} = \frac{1}{Q}\sum_{i=1}^{Q}\text{RR}_i
$$

where $Q$ is the number of queries.

Use MRR when the first correct result should appear as early as possible.


## Section 32 — Implement retrieval metrics

Now we implement the metrics manually.


In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    retrieved_at_k = retrieved_ids[:k]
    hits = set(retrieved_at_k).intersection(set(relevant_ids))
    return len(hits) / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    retrieved_at_k = retrieved_ids[:k]
    hits = set(retrieved_at_k).intersection(set(relevant_ids))
    return len(hits) / len(relevant_ids)


def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    retrieved_at_k = retrieved_ids[:k]
    hits = set(retrieved_at_k).intersection(set(relevant_ids))
    return 1 if len(hits) > 0 else 0


def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1 / rank
    return 0


## Explanation of the metric code

### `retrieved_ids[:k]`
Keeps only the top K returned documents.

### `set(...).intersection(...)`
Finds which retrieved IDs are also relevant.

Example:

```python
set([1, 4, 8]).intersection(set([1, 8]))
```

returns:

```text
{1, 8}
```

### `enumerate(retrieved_ids, start=1)`
Loops over retrieved documents while also tracking rank position.

We start rank from 1 because search results are ranked as first, second, third, etc.


## Section 33 — Evaluate one query

Let us evaluate the TF-IDF retriever for one query.


In [ ]:
query = "How do I reset my password?"

results = retrieve_top_k_tfidf(
    query=query,
    vectorizer=tfidf_vectorizer,
    document_matrix=tfidf_matrix,
    original_documents=documents,
    k=5
)

retrieved_ids = results["document_id"].tolist()
relevant_ids = ground_truth[query]

print("Query:", query)
print("Retrieved IDs:", retrieved_ids)
print("Relevant IDs:", relevant_ids)
print("Precision@3:", precision_at_k(retrieved_ids, relevant_ids, k=5))
print("Recall@3:", recall_at_k(retrieved_ids, relevant_ids, k=5))
print("Hit Rate@3:", hit_rate_at_k(retrieved_ids, relevant_ids, k=5))
print("Reciprocal Rank:", reciprocal_rank(retrieved_ids, relevant_ids))

results


Query: How do I reset my password?
Retrieved IDs: [1, 8, 0, 2, 3]
Relevant IDs: [1, 8]
Precision@3: 0.4
Recall@3: 1.0
Hit Rate@3: 1
Reciprocal Rank: 1.0


,document_id,document,score
0,1,Password reset requires entering your university email and verifying a code sent to your phone.,0.349902
1,8,Two-factor authentication protects accounts by requiring a password and a temporary security code.,0.173888
2,0,"The library account allows students to borrow books, renew loans, and reserve study rooms.",0.000000
3,2,"The writing center helps students improve essays, citations, and research proposals.",0.000000
4,3,"The campus health clinic provides vaccinations, first aid, and appointment booking.",0.000000


## Section 34 — Evaluate all queries

A system should not be judged using one query.

We need multiple queries because retrieval quality can vary by query type.


In [ ]:
def evaluate_tfidf_retriever(vectorizer, document_matrix, original_documents, ground_truth, k=3, max_results=10):
    rows = []

    for query, relevant_ids in ground_truth.items():
        results = retrieve_top_k_tfidf(
            query=query,
            vectorizer=vectorizer,
            document_matrix=document_matrix,
            original_documents=original_documents,
            k=max_results
        )

        retrieved_ids = results["document_id"].tolist()

        rows.append({
            "query": query,
            "retrieved_ids": retrieved_ids[:k],
            "relevant_ids": relevant_ids,
            f"precision@{k}": precision_at_k(retrieved_ids, relevant_ids, k),
            f"recall@{k}": recall_at_k(retrieved_ids, relevant_ids, k),
            f"hit_rate@{k}": hit_rate_at_k(retrieved_ids, relevant_ids, k),
            "reciprocal_rank": reciprocal_rank(retrieved_ids, relevant_ids)
        })

    return pd.DataFrame(rows)


In [ ]:
tfidf_evaluation_df = evaluate_tfidf_retriever(
    vectorizer=tfidf_vectorizer,
    document_matrix=tfidf_matrix,
    original_documents=documents,
    ground_truth=ground_truth,
    k=3,
    max_results=len(documents)
)

tfidf_evaluation_df


,query,retrieved_ids,relevant_ids,precision@3,recall@3,hit_rate@3,reciprocal_rank
0,How do I reset my password?,"[1, 8, 0]","[1, 8]",0.666667,1.0,1,1.0
1,Where can I borrow books?,"[0, 4, 1]",[0],0.333333,1.0,1,1.0
2,How can I get a refund?,"[4, 0, 1]",[4],0.333333,1.0,1,1.0
3,What workshop teaches machine learning?,"[5, 6, 0]",[5],0.333333,1.0,1,1.0
4,Where can I find journal articles?,"[7, 4, 0]",[7],0.333333,1.0,1,1.0
5,How do I print in color?,"[9, 0, 1]",[9],0.333333,1.0,1,1.0
6,What office helps with internships?,"[11, 2, 0]",[11],0.333333,1.0,1,1.0
7,What protects my account with a security code?,"[8, 0, 1]","[8, 1]",0.666667,1.0,1,1.0
8,Where can I check exam weeks and holidays?,"[10, 4, 9]",[10],0.333333,1.0,1,1.0
9,What workshop covers sensors and motors?,"[6, 5, 9]",[6],0.333333,1.0,1,1.0


In [ ]:
tfidf_summary = tfidf_evaluation_df[[
    "precision@3", "recall@3", "hit_rate@3", "reciprocal_rank"
]].mean()

tfidf_summary


precision@3        0.4
recall@3           1.0
hit_rate@3         1.0
reciprocal_rank    1.0
dtype: float64

## How to interpret average scores

Average Precision@3 tells us how clean the top 3 results are across queries.

Average Recall@3 tells us how many relevant documents are found across queries.

Average Hit Rate@3 tells us how often at least one correct document appears in the top 3.

Average Reciprocal Rank is MRR.

If MRR is close to 1, the first relevant result usually appears very early.


## Section 35 — Preprocessing profiles for retrieval

Now we test whether preprocessing helps or hurts retrieval.

Important rule:

> Preprocessing is not automatically good. It must be evaluated.

We will compare four profiles:

| Profile | What it does |
|---|---|
| `raw` | No cleaning |
| `minimal_clean` | Lowercase and normalize spaces |
| `punctuation_removed` | Minimal clean + remove punctuation |
| `aggressive_clean` | Remove punctuation and digits |

The aggressive profile can be dangerous because digits such as `14` may be meaningful.


In [ ]:
def minimal_clean(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def punctuation_removed(text):
    text = minimal_clean(text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def aggressive_clean(text):
    text = punctuation_removed(text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


preprocessing_profiles = {
    "raw": lambda x: x,
    "minimal_clean": minimal_clean,
    "punctuation_removed": punctuation_removed,
    "aggressive_clean": aggressive_clean
}


## Explanation of preprocessing functions

### `minimal_clean`
Lowercases text and removes repeated spaces.

Use when you want light normalization without destroying information.

### `punctuation_removed`
Removes punctuation characters.

Use when punctuation is not important.

Danger:

- `two-factor` becomes `two factor`, which may be fine.
- `C++` becomes `c`, which is bad.

### `aggressive_clean`
Removes digits.

Use only when numbers are definitely noise.

Danger:

- `14 days` loses `14`.
- `ISO 9001` loses `9001`.
- `2FA` may be damaged.


In [ ]:
def evaluate_profile(profile_name, cleaning_function, documents, ground_truth, k=3):
    processed_documents = [cleaning_function(doc) for doc in documents]

    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(processed_documents)

    rows = []

    for query, relevant_ids in ground_truth.items():
        processed_query = cleaning_function(query)

        results = retrieve_top_k_tfidf(
            query=processed_query,
            vectorizer=vectorizer,
            document_matrix=matrix,
            original_documents=processed_documents,
            k=len(processed_documents)
        )

        retrieved_ids = results["document_id"].tolist()

        rows.append({
            "profile": profile_name,
            "query": query,
            "retrieved_ids": retrieved_ids[:k],
            f"precision@{k}": precision_at_k(retrieved_ids, relevant_ids, k),
            f"recall@{k}": recall_at_k(retrieved_ids, relevant_ids, k),
            f"hit_rate@{k}": hit_rate_at_k(retrieved_ids, relevant_ids, k),
            "reciprocal_rank": reciprocal_rank(retrieved_ids, relevant_ids),
            "vocabulary_size": len(vectorizer.get_feature_names_out())
        })

    return pd.DataFrame(rows)


In [ ]:
profile_results = []

for profile_name, cleaning_function in preprocessing_profiles.items():
    profile_results.append(
        evaluate_profile(
            profile_name=profile_name,
            cleaning_function=cleaning_function,
            documents=documents,
            ground_truth=ground_truth,
            k=3
        )
    )

profile_comparison_df = pd.concat(profile_results, ignore_index=True)
profile_comparison_df.head(12)


,profile,query,retrieved_ids,precision@3,recall@3,hit_rate@3,reciprocal_rank,vocabulary_size
0,raw,How do I reset my password?,"[1, 8, 0]",0.666667,1.0,1,1.0,116
1,raw,Where can I borrow books?,"[0, 4, 1]",0.333333,1.0,1,1.0,116
2,raw,How can I get a refund?,"[4, 0, 1]",0.333333,1.0,1,1.0,116
3,raw,What workshop teaches machine learning?,"[5, 6, 0]",0.333333,1.0,1,1.0,116
4,raw,Where can I find journal articles?,"[7, 4, 0]",0.333333,1.0,1,1.0,116
5,raw,How do I print in color?,"[9, 0, 1]",0.333333,1.0,1,1.0,116
6,raw,What office helps with internships?,"[11, 2, 0]",0.333333,1.0,1,1.0,116
7,raw,What protects my account with a security code?,"[8, 0, 1]",0.666667,1.0,1,1.0,116
8,raw,Where can I check exam weeks and holidays?,"[10, 4, 9]",0.333333,1.0,1,1.0,116
9,raw,What workshop covers sensors and motors?,"[6, 5, 9]",0.333333,1.0,1,1.0,116


In [ ]:
profile_summary_df = profile_comparison_df.groupby("profile")[[
    "precision@3", "recall@3", "hit_rate@3", "reciprocal_rank", "vocabulary_size"
]].mean().sort_values(by="reciprocal_rank", ascending=False)

profile_summary_df


,precision@3,recall@3,hit_rate@3,reciprocal_rank,vocabulary_size
profile,,,,,
aggressive_clean,0.4,1.0,1.0,1.0,115.0
minimal_clean,0.4,1.0,1.0,1.0,116.0
punctuation_removed,0.4,1.0,1.0,1.0,116.0
raw,0.4,1.0,1.0,1.0,116.0


## How to interpret preprocessing comparison

Look for two things:

1. Retrieval metrics.
2. Vocabulary size.

A smaller vocabulary is not automatically better.

A larger vocabulary is not automatically better.

What matters is retrieval performance and error behavior.

Professional conclusion should be based on evidence, not habit.


## Section 36 — Lexical retrieval failure demonstration

Lexical retrieval fails when the query and document use different words.

Example:

```text
Query: How can I get my money back?
Relevant document: Students can request a tuition refund within 14 days of course withdrawal.
```

A human understands that "money back" and "refund" are related.

A lexical retriever may not.


In [ ]:
failure_query = "How can I get my money back?"

inspect_query_terms(failure_query, TfidfVectorizer(stop_words="english").fit(documents))


,term,in_vocabulary
0,money,False


In [ ]:
stopword_tfidf = TfidfVectorizer(stop_words="english")
stopword_tfidf_matrix = stopword_tfidf.fit_transform(documents)

retrieve_top_k_tfidf(
    query=failure_query,
    vectorizer=stopword_tfidf,
    document_matrix=stopword_tfidf_matrix,
    original_documents=documents,
    k=5
)


,document_id,document,score
0,0,"The library account allows students to borrow books, renew loans, and reserve study rooms.",0.0
1,1,Password reset requires entering your university email and verifying a code sent to your phone.,0.0
2,2,"The writing center helps students improve essays, citations, and research proposals.",0.0
3,3,"The campus health clinic provides vaccinations, first aid, and appointment booking.",0.0
4,4,Students can request a tuition refund within 14 days of course withdrawal.,0.0


## Interpretation of the failure case

When important query terms are missing from the vocabulary, the query vector may become weak or even all zeros.

Then similarity scores become unhelpful.

This is not solved by more cleaning.

This is a representation problem.

Later, semantic embeddings can help because they can map related meanings closer together.


## Section 37 — BM25 concept

BM25 is a strong lexical ranking method.

It is still keyword-based, but it improves over basic TF-IDF ranking by considering:

1. term frequency,
2. inverse document frequency,
3. document length normalization,
4. term frequency saturation.

### Term frequency saturation

If a word appears once, that is useful.

If it appears five times, that may be more useful.

But if it appears fifty times, it should not make the document fifty times better.

BM25 controls this using a saturation formula.


## Section 38 — BM25 math

For a query $Q$ and document $D$:

$$
\text{BM25}(D,Q) = \sum_{q_i \in Q} \text{IDF}(q_i) \cdot
\frac{f(q_i,D)(k_1+1)}{f(q_i,D) + k_1\left(1-b+b\frac{|D|}{\text{avgdl}}\right)}
$$

Where:

| Symbol | Meaning |
|---|---|
| $q_i$ | a query term |
| $f(q_i,D)$ | frequency of query term in document |
| D | document length |
| $\text{avgdl}$ | average document length in the corpus |
| $k_1$ | controls term-frequency saturation |
| $b$ | controls document-length normalization |

Common values:

```text
k1 = 1.5
b = 0.75
```

### BM25 IDF variant

We will use this common IDF variant:

$$
\text{IDF}(t) = \log\left(1 + \frac{N - \text{DF}(t) + 0.5}{\text{DF}(t) + 0.5}\right)
$$


## Section 39 — Implement BM25




In [ ]:
def simple_tokenize(text):
    # Lowercase text and extract alphanumeric tokens.
    return re.findall(r"[a-z0-9]+", text.lower())


def build_bm25_index(documents):
    tokenized_documents = [
        simple_tokenize(document)
        for document in documents
    ]

    bm25 = BM25Okapi(tokenized_documents)

    return bm25, tokenized_documents

def retrieve_top_k_bm25(query, documents, bm25, k=3):

    tokenized_query = simple_tokenize(query)

    scores = bm25.get_scores(tokenized_query)

    results = pd.DataFrame({
        "document_id": range(len(documents)),
        "document": documents,
        "score": scores
    })

    results = results.sort_values(
        by="score",
        ascending=False
    )

    return results.head(k).reset_index(drop=True)

In [ ]:
bm25, tokenized_documents = build_bm25_index(documents)
retrieve_top_k_bm25(
    query="How can I reset my password?",
    documents=documents,
    bm25=bm25,
    k=3
)

,document_id,document,score
0,1,Password reset requires entering your university email and verifying a code sent to your phone.,3.237021
1,4,Students can request a tuition refund within 14 days of course withdrawal.,2.104077
2,8,Two-factor authentication protects accounts by requiring a password and a temporary security code.,1.382892


## Section 40 — Evaluate BM25

Now we evaluate BM25 using the same ground truth and same metrics.


In [ ]:
def evaluate_bm25_retriever(documents, bm25, ground_truth, k=3, max_results=10):
    rows = []

    for query, relevant_ids in ground_truth.items():

        results = retrieve_top_k_bm25(
            query=query,
            documents=documents,
            bm25=bm25,
            k=max_results
        )

        retrieved_ids = results["document_id"].tolist()

        rows.append({
            "query": query,
            "retrieved_ids": retrieved_ids[:k],
            "relevant_ids": relevant_ids,
            f"precision@{k}": precision_at_k(
                retrieved_ids=retrieved_ids,
                relevant_ids=relevant_ids,
                k=k
            ),
            f"recall@{k}": recall_at_k(
                retrieved_ids=retrieved_ids,
                relevant_ids=relevant_ids,
                k=k
            ),
            f"hit_rate@{k}": hit_rate_at_k(
                retrieved_ids=retrieved_ids,
                relevant_ids=relevant_ids,
                k=k
            ),
            "reciprocal_rank": reciprocal_rank(
                retrieved_ids=retrieved_ids,
                relevant_ids=relevant_ids
            )
        })

    return pd.DataFrame(rows)

In [ ]:
# Build the BM25 index using the rank-bm25 library
bm25, tokenized_documents = build_bm25_index(documents)

# Evaluate the BM25 retriever
bm25_evaluation_df = evaluate_bm25_retriever(
    documents=documents,
    bm25=bm25,
    ground_truth=ground_truth,
    k=3,
    max_results=len(documents)
)

bm25_evaluation_df

,query,retrieved_ids,relevant_ids,precision@3,recall@3,hit_rate@3,reciprocal_rank
0,How do I reset my password?,"[1, 8, 0]","[1, 8]",0.666667,1.0,1,1.0
1,Where can I borrow books?,"[0, 4, 1]",[0],0.333333,1.0,1,1.0
2,How can I get a refund?,"[4, 8, 1]",[4],0.333333,1.0,1,1.0
3,What workshop teaches machine learning?,"[5, 6, 0]",[5],0.333333,1.0,1,1.0
4,Where can I find journal articles?,"[7, 4, 0]",[7],0.333333,1.0,1,1.0
5,How do I print in color?,"[9, 0, 1]",[9],0.333333,1.0,1,1.0
6,What office helps with internships?,"[11, 2, 0]",[11],0.333333,1.0,1,1.0
7,What protects my account with a security code?,"[8, 1, 0]","[8, 1]",0.666667,1.0,1,1.0
8,Where can I check exam weeks and holidays?,"[10, 4, 9]",[10],0.333333,1.0,1,1.0
9,What workshop covers sensors and motors?,"[6, 5, 9]",[6],0.333333,1.0,1,1.0


In [ ]:
bm25_summary = bm25_evaluation_df[[
    "precision@3", "recall@3", "hit_rate@3", "reciprocal_rank"
]].mean()

bm25_summary


precision@3        0.4
recall@3           1.0
hit_rate@3         1.0
reciprocal_rank    1.0
dtype: float64

## Section 41 — Compare TF-IDF and BM25

We will compare average metrics side by side.


In [ ]:
comparison = pd.DataFrame({
    "tfidf_cosine": tfidf_summary,
    "bm25": bm25_summary
})

comparison


,tfidf_cosine,bm25
precision@3,0.4,0.4
recall@3,1.0,1.0
hit_rate@3,1.0,1.0
reciprocal_rank,1.0,1.0


## How to interpret TF-IDF vs BM25 comparison

If BM25 performs better, likely reasons include:

- better handling of document length,
- better term-frequency saturation,
- stronger lexical ranking behavior.

If TF-IDF performs similarly, that is normal on small clean datasets.

Do not overgeneralize from one small corpus.

Professional rule:

> Compare retrievers on realistic data and realistic queries before choosing one.


## Section 42 — Chunking concept

In real retrieval systems, documents are often long.

Instead of retrieving a full document, systems usually retrieve smaller chunks.

A **chunk** is a piece of a larger document.

Why chunking matters:

1. Long documents may contain many unrelated topics.
2. The answer may exist in only one small section.
3. Smaller chunks can make retrieval more focused.
4. Very small chunks may lose context.
5. Very large chunks may add noise.

Chunking is a major design choice in retrieval and RAG systems.


## Section 43 — Chunking parameters

Two important parameters:

### `chunk_size`
Controls how large each chunk is.

In this lab, chunk size means number of words.

### `overlap`
Controls how many words are repeated between consecutive chunks.

Overlap helps avoid losing information at chunk boundaries.

Example:

```text
Chunk 1: password reset requires university email and verification code
Chunk 2: verification code sent to your phone before account access
```

The phrase `verification code` appears in both chunks, so the idea is preserved.


In [ ]:
long_document = '''
The student services handbook explains library access, password reset, account protection,
printing payments, health clinic appointments, academic calendar deadlines, internship advising,
and research database access. Students should use their university email for account recovery.
Password reset requires a verification code. The academic calendar lists exam weeks, holidays,
registration dates, and withdrawal deadlines. The library account supports book borrowing,
loan renewal, and study room reservations. The printing service supports color printing,
black-and-white printing, scanning, and payment by student card.
'''


def chunk_text_by_words(text, chunk_size=35, overlap=8):
    words = text.split()

    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start = start + chunk_size - overlap

    return chunks


chunks = chunk_text_by_words(long_document, chunk_size=35, overlap=8)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}")
    print(chunk)
    print("-" * 100)


Chunk 0
The student services handbook explains library access, password reset, account protection, printing payments, health clinic appointments, academic calendar deadlines, internship advising, and research database access. Students should use their university email for account recovery. Password
----------------------------------------------------------------------------------------------------
Chunk 1
use their university email for account recovery. Password reset requires a verification code. The academic calendar lists exam weeks, holidays, registration dates, and withdrawal deadlines. The library account supports book borrowing, loan renewal, and study
----------------------------------------------------------------------------------------------------
Chunk 2
account supports book borrowing, loan renewal, and study room reservations. The printing service supports color printing, black-and-white printing, scanning, and payment by student card.
------------------------------------

## Explanation of the chunking code

### `text.split()`
Splits the text into words using whitespace.

This is simple but not perfect. Production systems usually use better tokenizers.

### `chunk_size`
Controls the maximum number of words per chunk.

### `overlap`
Controls how many words are reused from the previous chunk.

### Error checks
We reject invalid settings:

- `chunk_size <= 0`
- `overlap < 0`
- `overlap >= chunk_size`

These checks prevent infinite loops and broken chunks.


## Section 44 — Retrieve from chunks

Now we treat chunks as retrievable documents.

This is the bridge toward retrieval-augmented systems.


In [ ]:
chunk_vectorizer = TfidfVectorizer()
chunk_matrix = chunk_vectorizer.fit_transform(chunks)

retrieve_top_k_tfidf(
    query="How do I reset my password?",
    vectorizer=chunk_vectorizer,
    document_matrix=chunk_matrix,
    original_documents=chunks,
    k=3
)


,document_id,document,score
0,0,"The student services handbook explains library access, password reset, account protection, printing payments, health clinic appointments, academic calendar ...",0.294916
1,1,"use their university email for account recovery. Password reset requires a verification code. The academic calendar lists exam weeks, holidays, registration...",0.216864
2,2,"account supports book borrowing, loan renewal, and study room reservations. The printing service supports color printing, black-and-white printing, scanning...",0.000000


## Interpretation

The retriever now returns chunks instead of full documents.

This is important because the most relevant information may be inside one part of a long document.

In later systems, the retrieved chunks are passed to a language model as context.

But the retrieval step itself is what you learned here.


## Section 45 — Error analysis

Metrics tell you the score.

Error analysis tells you why the score happened.

For every failed query, ask:

1. What was the query?
2. What should have been retrieved?
3. What was retrieved instead?
4. Which query terms matched?
5. Which important terms were missing?
6. Did preprocessing damage the text?
7. Was the failure caused by vocabulary mismatch?
8. Was the query ambiguous?
9. Is the relevant document missing necessary wording?

Professional retrieval work is mostly debugging failures.


In [ ]:
def analyze_query(query, relevant_ids, vectorizer, document_matrix, documents, k=3):
    results = retrieve_top_k_tfidf(
        query=query,
        vectorizer=vectorizer,
        document_matrix=document_matrix,
        original_documents=documents,
        k=k
    )

    retrieved_ids = results["document_id"].tolist()

    print("Query:", query)
    print("Relevant IDs:", relevant_ids)
    print("Retrieved IDs:", retrieved_ids)
    print("Precision@3:", precision_at_k(retrieved_ids, relevant_ids, 3))
    print("Recall@3:", recall_at_k(retrieved_ids, relevant_ids, 3))
    print("Hit Rate@3:", hit_rate_at_k(retrieved_ids, relevant_ids, 3))
    print("Reciprocal Rank:", reciprocal_rank(retrieved_ids, relevant_ids))
    print("\nQuery term vocabulary check:")
    display(inspect_query_terms(query, vectorizer))

    return results


In [ ]:
analyze_query(
    query="How can I get my money back?",
    relevant_ids=[4],
    vectorizer=stopword_tfidf,
    document_matrix=stopword_tfidf_matrix,
    documents=documents,
    k=3
)


Query: How can I get my money back?
Relevant IDs: [4]
Retrieved IDs: [0, 1, 2]
Precision@3: 0.0
Recall@3: 0.0
Hit Rate@3: 0
Reciprocal Rank: 0

Query term vocabulary check:


,term,in_vocabulary
0,money,False


,document_id,document,score
0,0,"The library account allows students to borrow books, renew loans, and reserve study rooms.",0.0
1,1,Password reset requires entering your university email and verifying a code sent to your phone.,0.0
2,2,"The writing center helps students improve essays, citations, and research proposals.",0.0


## Section 46 — Final student assignment

You must build and evaluate your own retrieval system.

Do not copy the exact corpus from this notebook. Create your own neutral knowledge base.

Possible themes:

- university services,
- hospital FAQ,
- library help center,
- travel policy,
- software documentation,
- product support articles,
- environmental policy guide.


## Assignment requirements

### Task 1 — Build a corpus

Create at least 15 documents.

Each document should be 1 to 3 sentences.

### Task 2 — Build queries

Create at least 10 user queries.

Queries should include:

- direct keyword queries,
- full natural-language questions,
- at least two queries with vocabulary mismatch.

### Task 3 — Create ground truth

For each query, define the relevant document IDs.

### Task 4 — Build a Bag-of-Words representation

Use `CountVectorizer` and inspect:

- vocabulary size,
- matrix shape,
- sparsity percentage,
- non-zero terms for one document.

### Task 5 — Build a TF-IDF retriever

Use:

- `TfidfVectorizer`,
- `transform()` for queries,
- `cosine_similarity`,
- a reusable `retrieve_top_k` function.

### Task 6 — Evaluate retrieval

Calculate:

- Precision@3,
- Recall@3,
- Hit Rate@3,
- MRR.

### Task 7 — Compare preprocessing profiles

Compare:

- raw text,
- minimal cleaning,
- punctuation removal,
- aggressive cleaning.

### Task 8 — Compare TF-IDF with BM25

Use the BM25 implementation from this notebook.

### Task 9 — Perform error analysis

Choose at least 3 failed or weak queries and explain the failure reason.


## Final takeaways

1. Preprocessing is not the final goal.
2. Retrieval starts when text becomes vectors.
3. Bag-of-Words is simple but limited.
4. TF-IDF improves raw counts by weighting rare terms more strongly.
5. Cosine similarity compares query and document vectors.
6. Top-k retrieval is a ranking problem.
7. Metrics require ground truth.
8. Precision@K and Recall@K measure different things.
9. BM25 is a stronger lexical baseline than basic TF-IDF.
10. Chunking is essential before building retrieval over long documents.
11. Lexical retrieval fails under vocabulary mismatch.
12. The next natural step is semantic embeddings.

A serious retrieval engineer does not only build a retriever.

A serious retrieval engineer evaluates it, breaks it, explains the failures, and improves it.


## Official documentation references

Use these references when you want to review the tools used in this lab:

- scikit-learn `CountVectorizer`: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
- scikit-learn `TfidfVectorizer`: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- scikit-learn `cosine_similarity`: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
- Python `re`: https://docs.python.org/3/library/re.html
- pandas documentation: https://pandas.pydata.org/docs/
- NumPy documentation: https://numpy.org/doc/
